# MLflow — Experiment Tracking & Model Registry

**Track, compare, and reproduce** your ML experiments.

- **Tracking**: Log parameters, metrics, artifacts
- **Projects**: Reproducible packaging
- **Registry**: Version and stage models
- **Serving**: Deploy models as REST APIs

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    import mlflow
    import mlflow.sklearn
    print(f'MLflow: {mlflow.__version__}')
except ImportError:
    print('Install: pip install mlflow')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Dataset
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Dataset: {X.shape} | Classes: {np.unique(y)}')

## 1. MLflow Basics — Setting Up Tracking

In [ ]:
# Set experiment name
mlflow.set_experiment('Wine-Classification-Comparison')

# Simple run example
with mlflow.start_run(run_name='Quick-RF-Test'):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    
    # Log parameters
    mlflow.log_param('model', 'RandomForest')
    mlflow.log_param('n_estimators', 100)
    
    # Log metrics
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1_weighted', f1_score(y_test, rf.predict(X_test), average='weighted'))
    
    # Log model
    mlflow.sklearn.log_model(rf, 'model')
    
    print(f'Logged run with accuracy: {acc:.4f}')
    print(f'Run ID: {mlflow.active_run().info.run_id}')

## 2. Systematic Model Comparison

In [ ]:
# Define experiments to run
experiments = [
    {
        'name': 'LogisticRegression',
        'model': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
        'params': {'C': 1.0, 'solver': 'lbfgs'}
    },
    {
        'name': 'RandomForest_100',
        'model': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
        'params': {'n_estimators': 100, 'max_depth': 5}
    },
    {
        'name': 'RandomForest_300',
        'model': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
        'params': {'n_estimators': 300, 'max_depth': 10}
    },
    {
        'name': 'GradientBoosting',
        'model': GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42),
        'params': {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}
    },
    {
        'name': 'SVM_RBF',
        'model': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=10))]),
        'params': {'kernel': 'rbf', 'C': 10}
    }
]

results = []

for exp in experiments:
    with mlflow.start_run(run_name=exp['name']):
        # Train
        model = exp['model']
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        
        # Metrics
        acc = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, average='weighted')
        cv_score = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
        
        # Log everything
        mlflow.log_param('model_name', exp['name'])
        for k, v in exp['params'].items():
            mlflow.log_param(k, v)
        
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('f1_weighted', f1)
        mlflow.log_metric('cv_accuracy', cv_score)
        
        mlflow.sklearn.log_model(model, 'model')
        
        results.append({'Model': exp['name'], 'Accuracy': acc, 'F1': f1, 'CV Score': cv_score})
        print(f'{exp["name"]:25s} | Acc: {acc:.4f} | F1: {f1:.4f} | CV: {cv_score:.4f}')

results_df = pd.DataFrame(results)
print(f'\nAll experiments logged to MLflow!')

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = range(len(results_df))
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6']

bars = axes[0].bar(x_pos, results_df['Accuracy'], color=colors, edgecolor='black')
for bar, val in zip(bars, results_df['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.3f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_df['Model'], rotation=30, ha='right')
axes[0].set_title('Test Accuracy'); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x_pos, results_df['CV Score'], color=colors, edgecolor='black')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(results_df['Model'], rotation=30, ha='right')
axes[1].set_title('Cross-Validation Score'); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('MLflow Experiment Comparison — Wine Classification', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Loading & Using Logged Models

In [ ]:
# Query MLflow for best run
experiment = mlflow.get_experiment_by_name('Wine-Classification-Comparison')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                              order_by=['metrics.accuracy DESC'])
    print('Top 3 runs by accuracy:')
    print(runs[['tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1_weighted']].head(3).to_string())
    
    # Load best model
    best_run_id = runs.iloc[0]['run_id']
    best_model = mlflow.sklearn.load_model(f'runs:/{best_run_id}/model')
    print(f'\nLoaded best model from run: {best_run_id[:8]}...')
    print(f'Prediction on test: {best_model.predict(X_test[:5])}')

## 4. Log Artifacts (Plots, Reports)

In [ ]:
# Log a plot as artifact
with mlflow.start_run(run_name='Best-Model-With-Report'):
    best_exp = experiments[results_df['Accuracy'].idxmax()]
    model = best_exp['model']
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mlflow.log_metric('accuracy', accuracy_score(y_test, preds))
    
    # Save classification report as artifact
    report = classification_report(y_test, preds, target_names=wine.target_names)
    with open('classification_report.txt', 'w') as f:
        f.write(report)
    mlflow.log_artifact('classification_report.txt')
    
    # Save confusion matrix plot
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=wine.target_names, ax=ax, cmap='Blues')
    plt.title(f'Best Model: {best_exp["name"]}')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=100)
    mlflow.log_artifact('confusion_matrix.png')
    plt.show()
    
    print(f'Artifacts logged!')
    print(f'\n{report}')

## 5. MLflow Cheat Sheet

```python
# Setup
mlflow.set_tracking_uri('http://localhost:5000')  # Remote server
mlflow.set_experiment('my-experiment')

# Logging
with mlflow.start_run():
    mlflow.log_param('lr', 0.01)           # Single parameter
    mlflow.log_params({'a': 1, 'b': 2})    # Multiple params
    mlflow.log_metric('rmse', 0.5)         # Single metric
    mlflow.log_metric('loss', 0.3, step=1) # Metric with step
    mlflow.log_artifact('plot.png')        # File artifact
    mlflow.sklearn.log_model(model, 'model')  # Model

# Auto-logging (no manual logging needed!)
mlflow.sklearn.autolog()  # Works for sklearn, TF, PyTorch, XGBoost

# Loading
model = mlflow.sklearn.load_model('runs:/<run_id>/model')

# UI: mlflow ui  (in terminal, opens http://localhost:5000)
```

### MLflow Components:

| Component | Purpose |
|-----------|--------|
| **Tracking** | Log params, metrics, models |
| **Projects** | Reproducible packaging (conda/docker) |
| **Models** | Standard model format, multi-framework |
| **Registry** | Version, stage (Staging → Production) |
| **Serving** | Deploy as REST endpoint |

### Tips:
- Use `mlflow.autolog()` for quick experiments
- Always log the **random_state** for reproducibility
- Use **tags** to organize runs (`mlflow.set_tag('team', 'data-science')`)
- Run `mlflow ui` to view experiments in a nice web interface